In [12]:
import kaggle_benchmarks as kbench
from kaggle_benchmarks.prompting import ResponseParsingError
from pydantic import BaseModel, Field

class BinaryQA(BaseModel):
    question: str = Field(description="The generated binary question.")
    answer: str = Field(description="The answer to the question, must be one of 'Yes', 'No', or 'Unknown'.")

@kbench.task(name="rewrite_conclusion_as_binary_qa")
def rewrite_conclusion_as_binary_qa(llm,conclusion = "This two-stage IPD meta-analysis provides robust evidence of a bidirectional relationship between CLD and OP in older adults. These findings underscore the need for integrated screening and management of both conditions in aging populations."
):
    
    prompt = f"""
    Please analyze the following scientific conclusion and rephrase it as a single, clear binary question and a one-word answer from the options: Yes, No, or Unknown.

    Conclusion: "{conclusion}"

    Your output must be a JSON object with two keys: "question" and "answer".
    """

    try:
        response = llm.prompt(prompt, schema=BinaryQA)

        kbench.assertions.assert_contains_regex(
            r"(?i)^(yes|no|unknown)",
            response.answer.strip(),
           # expectation="The answer should be 'Yes' based on the provided text."
        )

    except ResponseParsingError as e:
        kbench.assertions.assert_fail(
            expectation=f"The model's output did not conform to the BinaryQA schema. Error: {e.error}"
        )



In [24]:
import pandas as pd # Optional, but great for viewing the final list

# 1. Define your list of scientific statements
my_statements = [
    "Plasma cfChIP-seq captures hepatocyte disease-specific gene activity in AIH patients.",
    "Daily consumption of green tea significantly reduces the risk of cardiovascular disease in adults over 50.",
    "The experimental vaccine did not show a statistically significant improvement in neutralizing the virus compared to the placebo."
]

In [25]:


# 2. Create an empty list to store your successfully extracted data
qa_results = []

# 3. Loop through each statement
for i, statement in enumerate(my_statements):
    print(f"Processing statement {i+1}/{len(my_statements)}...")
    
    # Run the LLM task
    run_result = rewrite_conclusion_as_binary_qa.run(kbench.llm, conclusion=statement)
    
    try:
        # First, ensure the kbench task actually succeeded
        if run_result.status.name == 'SUCCESS':
            
            # Extract the Pydantic object using the path we found earlier
            llm_output = run_result.chat.history[1].content
            
            # Store the data in a standard Python dictionary
            qa_results.append({
                "Original Statement": statement,
                "Question": llm_output.question,
                "Answer": llm_output.answer
            })
            print(f"  ✓ Success: Answered '{llm_output.answer}'")
            
        else:
            print(f"  ✗ Task failed or was rejected. Status: {run_result.status}")
            
    except (IndexError, AttributeError) as e:
        # This catches edge cases where the chat history is empty or formatted weirdly
        print(f"  ✗ Failed to parse output structure. Error: {e}")

# 4. View the results (using Pandas makes it look like a nice table)
print("\n--- Final Extracted Data ---")
df_results = pd.DataFrame(qa_results)
print(df_results)

Processing statement 1/3...
  ✓ Success: Answered 'Yes'
Processing statement 2/3...
  ✓ Success: Answered 'Yes'
Processing statement 3/3...
  ✓ Success: Answered 'No'

--- Final Extracted Data ---
                                  Original Statement  \
0  Plasma cfChIP-seq captures hepatocyte disease-...   
1  Daily consumption of green tea significantly r...   
2  The experimental vaccine did not show a statis...   

                                            Question Answer  
0  Does Plasma cfChIP-seq capture hepatocyte dise...    Yes  
1  Does daily consumption of green tea significan...    Yes  
2  Did the experimental vaccine show a statistica...     No  


In [21]:
# 1. Access the second message in the chat history (the LLM's response)
llm_message = qa1.chat.history[1]

# 2. The 'content' of this message is your BinaryQA Pydantic model
binary_qa_result = llm_message.content

# 3. Now you can access the attributes normally!
extracted_question = binary_qa_result.question
extracted_answer = binary_qa_result.answer

print("Question:", extracted_question)
print("Answer:", extracted_answer)

Question: Does plasma cfChIP-seq offer a non-invasive and accurate method for diagnosing and monitoring AIH?
Answer: Yes
